# Étape 7 - Synthèse de modélisation et fiche modèle

## Objectif

La modélisation est faite : l'étape 4 a comparé six modèles à une référence naïve,
l'étape 5 a optimisé, évalué sur le jeu de test et interprété. **Ce notebook ne réentraîne
rien**, et c'est délibéré : réentraîner ici créerait un second modèle, donc une seconde
vérité, pour un projet qui n'en veut qu'une.

Il fait ce qui manquait entre le modèle et son usage :

1. **Vérifier la conformité** du modèle livré, par des tests qui échouent si elle est
   rompue, plutôt que par une affirmation dans un document.
2. **Livrer le code d'inférence** : un tiers doit pouvoir prédire sans rejouer un notebook.
3. **Produire la fiche modèle**, qui dit ce que le modèle fait, ce qu'il ne fait pas, et
   pour qui il peut être dangereux.

Protocole et résultats détaillés : [`étape_4.md`](../etapes/étape_4.md) et
[`étape_5.md`](../etapes/étape_5.md).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_PATH = Path.cwd().parent
sys.path.insert(0, str(PROJECT_PATH))

from modeles.inference import charger, predire, preparer_ligne_brute  # noqa: E402

DATA_PATH = PROJECT_PATH / 'csv' / 'dataset_phase3_final.csv'
CARTE_PATH = PROJECT_PATH / 'MODEL_CARD.md'

modele = charger()
df = pd.read_csv(DATA_PATH, dtype={'Code du diplôme SISE': str})

pipeline = modele['pipeline']
foret = pipeline.named_steps['modele']

print(f'Modèle : {type(foret).__name__}, {foret.n_estimators} arbres')
print(f'Variables : {len(modele["features"])}')
print(f'Cible : {modele["cible"]}')
print(f'Métriques de test (relevées à l\'étape 5) : {modele["metriques_test"]}')

Modèle : RandomForestRegressor, 200 arbres
Variables : 19
Cible : 6-Taux d'emploi salarié en France - 6 mois après le diplôme
Métriques de test (relevées à l'étape 5) : {'MAE': 9.579, 'RMSE': 12.373, 'R²': 0.554}


---

## 1. Le protocole, en une page

| Décision | Ce qui a été fait | Où |
|---|---|---|
| Périmètre | 17 065 formations, 19 variables explicatives, 8 catégorielles et 11 numériques | Étape 3 |
| Découpage | 80 / 20, `random_state=42`, **avant** tout prétraitement | Étape 3 |
| Prétraitement | One-hot des catégorielles et passthrough des numériques, **dans la pipeline** | Étape 4 |
| Référence | `DummyRegressor(strategy='mean')`, évaluée dans les mêmes conditions | Étape 4 |
| Comparaison | 6 modèles, validation croisée 5 plis, mêmes plis pour tous | Étape 4 |
| Sélection | Forêt aléatoire, choisie sur la stabilité entre plis à score égal | Étape 4 |
| Optimisation | `GridSearchCV`, `cv=5`, puis **règle à un écart-type** | Étape 5 |
| Évaluation | Jeu de test ouvert **une seule fois** : MAE 9,579, RMSE 12,373, R² 0,554 | Étape 5 |
| Interprétation | Importance par impureté et par permutation, confrontées à l'EDA | Étape 5 |
| Robustesse | Recouvrement des formations chiffré, validation temporelle 2023-2024 | Étape 5 |

**Le choix qui structure tout le reste** : la métrique principale est la MAE, en points de
taux d'emploi. Une erreur de 9,58 points se comprend sans traduction par un responsable de
formation, ce qui n'est pas le cas d'un R carré.

---

## 2. Contrôles de conformité

Les trois règles éliminatoires de la grille sont vérifiables sur le modèle livré, pas
seulement sur les notebooks. Ces contrôles portent sur ce qui a été **sauvegardé** : si un
jour quelqu'un réentraîne en introduisant une fuite, ils échouent.

In [2]:
MOTIFS_FUITE = ['12-', '18-', '24-', '30-', 'emploi stable', 'emploi non salarié']
IDENTIFIANTS = ["Code UAI de l'établissement", 'Code du diplôme SISE',
                'Établissement', 'Libellé du diplôme']

controles = []


def controler(intitule, condition, detail):
    """Enregistre un contrôle et échoue immédiatement s'il n'est pas satisfait."""
    controles.append({'contrôle': intitule, 'résultat': 'OK' if condition else 'ÉCHEC',
                      'détail': detail})
    assert condition, f'{intitule} : {detail}'


features = modele['features']

controler('La cible est absente des variables',
          modele['cible'] not in features,
          f'{len(features)} variables, aucune n\'est la cible')

fuitantes = [f for f in features if any(motif in f for motif in MOTIFS_FUITE)]
controler('Aucune variable postérieure à la cible',
          not fuitantes,
          'aucune variable mesurée à 12, 18, 24, 30 mois ni sur l\'emploi stable')

controler('Aucun identifiant à haute cardinalité',
          not [f for f in features if f in IDENTIFIANTS],
          'les codes UAI et SISE servent aux variables dérivées, jamais en entrée directe')

controler('Le prétraitement est dans la pipeline',
          [nom for nom, _ in pipeline.steps] == ['pretraitement', 'modele'],
          'une pipeline scikit-learn, donc réajustée sur le seul train à chaque pli')

controler('Le hasard est fixé',
          foret.random_state == 42,
          f'random_state={foret.random_state} sur la forêt')

# Déterminisme : deux appels sur les mêmes entrées, à la tolérance flottante près. La
# sommation parallèle des arbres interdit une égalité stricte, voir la lecture ci-dessous.
echantillon = df.head(200)
preparees = pd.concat(
    [preparer_ligne_brute(ligne, modele)[0] for _, ligne in echantillon.iterrows()],
    ignore_index=True)
premier = pipeline.predict(preparees)
second = pipeline.predict(preparees)
ecart_max = float(np.abs(premier - second).max())
controler('Les prédictions sont reproductibles',
          np.allclose(premier, second, rtol=1e-10, atol=1e-10),
          f'écart maximal de {ecart_max:.1e} point entre deux appels identiques')

controler('Les métriques de test sont livrées avec le modèle',
          set(modele['metriques_test']) == {'MAE', 'RMSE', 'R²'},
          'MAE, RMSE et R² voyagent avec le modèle, pas seulement dans un rapport')

controler('Le référentiel d\'inférence est embarqué',
          'defauts' in modele['referentiel'],
          'les valeurs de repli sont calculées sur le seul jeu d\'apprentissage')

print(f'{len(controles)} contrôles exécutés, tous passés.')
pd.DataFrame(controles).set_index('contrôle')

8 contrôles exécutés, tous passés.


,résultat,détail
contrôle,,
La cible est absente des variables,OK,"19 variables, aucune n'est la cible"
Aucune variable postérieure à la cible,OK,"aucune variable mesurée à 12, 18, 24, 30 mois ..."
Aucun identifiant à haute cardinalité,OK,les codes UAI et SISE servent aux variables dé...
Le prétraitement est dans la pipeline,OK,"une pipeline scikit-learn, donc réajustée sur ..."
Le hasard est fixé,OK,random_state=42 sur la forêt
Les prédictions sont reproductibles,OK,écart maximal de 3.6e-14 point entre deux appe...
Les métriques de test sont livrées avec le modèle,OK,"MAE, RMSE et R² voyagent avec le modèle, pas s..."
Le référentiel d'inférence est embarqué,OK,les valeurs de repli sont calculées sur le seu...


**Ce que ces contrôles garantissent, et ce qu'ils ne garantissent pas.**

Ils garantissent que le fichier livré est conforme : pas de cible ni de variable
postérieure dans les entrées, pas d'identifiant brut, un prétraitement enfermé dans la
pipeline, un hasard fixé, des prédictions reproductibles. Un `assert` échoue si l'une de ces
propriétés est perdue, ce qui les rend plus solides qu'une case cochée dans un document.

**Une nuance sur la reproductibilité.** Deux appels successifs sur les mêmes entrées ne
donnent pas des valeurs identiques au bit près, mais à environ 2 × 10⁻¹⁴ près. Ce n'est pas
un défaut de fixation du hasard : la forêt somme ses arbres en parallèle (`n_jobs=-1`), et
l'addition en virgule flottante n'est pas associative, donc l'ordre d'accumulation change le
dernier chiffre. L'écart est quinze ordres de grandeur sous la précision de la cible, qui se
mesure en points de pourcentage. Le contrôle teste donc l'égalité à la tolérance flottante,
et non une égalité stricte qui échouerait pour une raison sans rapport avec le modèle.

Ils ne garantissent pas que le jeu de test n'a servi qu'une fois : cela ne se lit pas dans
un fichier, cela se lit dans l'historique du projet. L'étape 4 supprime le jeu de test de
la mémoire dès la première cellule, l'étape 5 ne l'ouvre qu'à la section 3, et aucune
décision n'a été prise après. C'est une affirmation vérifiable en lisant les notebooks, pas
une propriété mesurable ici.

---

## 3. Le modèle est-il utilisable par un tiers ?

Un modèle qui ne s'utilise qu'en rejouant le notebook qui l'a produit n'est pas livré. Le
module [`modeles/inference.py`](../modeles/inference.py) prend une formation telle
qu'InserSup la publie, reconstruit les onze variables dérivées à partir du référentiel
embarqué, et renvoie une prédiction.

In [3]:
CAS_REFERENCE = {
    'Région': 'Bretagne', 'Académie': 'Rennes',
    'Type de diplôme': 'Licence professionnelle',
    'Domaine disciplinaire': 'Sciences, technologies, santé',
    'Discipline': 'Sciences fondamentales et applications',
    'Secteur disciplinaire': 'Informatique', 'Promotion': 2024,
    "Code UAI de l'établissement": '0350936C', 'Code du diplôme SISE': '99999',
    '6-Nombre de sortants - 6 mois après le diplôme': 48,
    '6-Nombre de poursuivants - 6 mois après le diplôme': 6,
}

# Non-régression : le module doit reproduire exactement la démonstration de l'étape 5.
reference = predire(modele, CAS_REFERENCE)
assert abs(reference['prediction'] - 68.4) < 0.05, (
    f"Le module d'inférence diverge de l'étape 5 : {reference['prediction']} au lieu de 68.4")
print(f"Non-régression vérifiée : {reference['prediction']} %, comme à l'étape 5")
print(f"Variables retombées sur un défaut : {', '.join(reference['replis'])}")
print()

# Trois cas contrastés, pour voir ce que le modèle fait varier.
variantes = {
    'Licence pro informatique (référence)': CAS_REFERENCE,
    'Même formation, en licence générale': {
        **CAS_REFERENCE, 'Type de diplôme': 'Licence générale'},
    'Master LMD en langues, promotion 2020': {
        **CAS_REFERENCE, 'Type de diplôme': 'Master LMD', 'Promotion': 2020,
        'Domaine disciplinaire': 'Lettres, langues et arts',
        'Discipline': 'Langues et littératures étrangères',
        'Secteur disciplinaire': 'Langues et littératures étrangères'},
}

resultats = []
for intitule, cas in variantes.items():
    sortie = predire(modele, cas)
    resultats.append({
        'cas': intitule,
        'prédiction': f"{sortie['prediction']} %",
        'intervalle': f"{sortie['intervalle'][0]} à {sortie['intervalle'][1]} %",
        'valeurs par défaut': len(sortie['replis']),
    })

pd.DataFrame(resultats).set_index('cas')

Non-régression vérifiée : 68.4 %, comme à l'étape 5
Variables retombées sur un défaut : nb_etablissements_par_diplome, nb_formations_etablissement, sortants_etablissement



,prédiction,intervalle,valeurs par défaut
cas,,,
Licence pro informatique (référence),68.4 %,58.8 à 78.0 %,3
"Même formation, en licence générale",53.0 %,43.4 à 62.6 %,3
"Master LMD en langues, promotion 2020",29.1 %,19.6 à 38.7 %,3


**Lecture.** Les trois cas ne diffèrent que par ce qu'on fait varier, et l'écart est celui
qu'annonçait l'analyse : passer d'une licence professionnelle à une licence générale, à
effectif, région et promotion identiques, fait chuter la prédiction. Le troisième cas cumule
les facteurs défavorables identifiés à l'étape 2, discipline littéraire et promotion 2020,
et se retrouve nettement plus bas.

**Une prédiction ne se lit jamais seule.** L'intervalle affiché vaut plus ou moins la MAE de
test, soit environ 9,6 points. Il est indicatif, pas statistique : ce n'est pas un intervalle
de confiance, c'est l'ordre de grandeur de l'erreur habituelle. Sur une formation de moins de
25 sortants, il faudrait le lire plus large encore, l'étape 5 y mesure 12,09 points de MAE.

**Les modalités inconnues ne font pas échouer la prédiction.** Le code SISE `99999` n'existe
pas dans le référentiel : les variables concernées retombent sur la médiane d'apprentissage,
et la fonction le signale plutôt que de le taire. C'est un choix : en production, une
formation nouvelle doit pouvoir être évaluée, à condition que l'utilisateur sache que trois
de ses variables sont des valeurs par défaut.

---

## 4. La fiche modèle

Le dernier livrable manquant. Elle est **générée** à partir du modèle chargé, pour qu'elle ne
puisse pas décrire autre chose que ce qui est réellement livré.

In [4]:
# Relevés de l'étape 5, cités comme tels : ils portent sur le jeu de test, que ce
# notebook n'ouvre pas.
RELEVES_ETAPE_5 = {
    'mae_reference_naive': 15.069,
    'r2_temporel': 0.458,
    'mae_petites_formations': 12.09,
    'mae_grandes_formations': 6.33,
    'recouvrement_formations': 90.0,
    'mae_formations_inconnues': 11.28,
    'lignes_test': 3413,
}

metriques = modele['metriques_test']
gain = RELEVES_ETAPE_5['mae_reference_naive'] - metriques['MAE']

carte = f"""# Fiche du modèle : insertion professionnelle des formations

*Fiche générée par [`etape_7_synthese_modele.ipynb`](notebooks/etape_7_synthese_modele.ipynb)
à partir du modèle livré. Ne pas modifier à la main.*

## Ce que fait ce modèle

Il prédit le **taux d'emploi salarié en France six mois après le diplôme** d'une formation
de l'enseignement supérieur, à partir de {len(modele['features'])} caractéristiques connues
avant l'enquête d'insertion.

- **Tâche** : régression, cible continue de 0 à 100 %.
- **Algorithme** : {type(foret).__name__}, {foret.n_estimators} arbres,
  `min_samples_leaf={foret.min_samples_leaf}`, `random_state={foret.random_state}`.
- **Unité de prédiction** : une formation, c'est-à-dire un diplôme d'un établissement pour
  une promotion donnée. **Jamais une personne.**

## Données d'entraînement

- **Source** : InserSup, millésime 2026_S1, ministère de l'Enseignement supérieur, Licence
  Ouverte Etalab.
- **Périmètre** : 17 065 formations, promotions 2019 à 2024, France.
- **Découpage** : 80 % apprentissage, 20 % test, `random_state=42`, effectué avant tout
  prétraitement.
- **Exclusions volontaires** : tous les indicateurs d'emploi mesurés à 12, 18, 24 et
  30 mois, ainsi que l'emploi stable et non salarié à 6 mois. Mesurés au même horizon que la
  cible ou après elle, ils constitueraient une fuite de données.

## Performance

| Métrique | Valeur | Lecture |
|---|---|---|
| MAE | **{metriques['MAE']} points** | Erreur moyenne sur {RELEVES_ETAPE_5['lignes_test']} formations jamais vues |
| RMSE | {metriques['RMSE']} points | Pénalise les grosses erreurs |
| R² | {metriques['R²']} | Part de la variance expliquée |
| Référence naïve | {RELEVES_ETAPE_5['mae_reference_naive']} points de MAE | Prédire la moyenne pour tout le monde |
| **Gain** | **{gain:.2f} points, soit {100 * gain / RELEVES_ETAPE_5['mae_reference_naive']:.0f} %** | Ce que le modèle apporte réellement |

Le jeu de test n'a été ouvert qu'une fois, après le choix définitif du modèle.

## Où ce modèle échoue

| Situation | Effet mesuré |
|---|---|
| Formations de 25 sortants ou moins | MAE de {RELEVES_ETAPE_5['mae_petites_formations']} points, contre {RELEVES_ETAPE_5['mae_grandes_formations']} au-delà de 90 sortants |
| Formation jamais vue à l'entraînement | MAE de {RELEVES_ETAPE_5['mae_formations_inconnues']} points, soit +1,89 point |
| Prédire une promotion future | R² de {RELEVES_ETAPE_5['r2_temporel']} en entraînant sur 2019-2022 et testant sur 2023-2024 |
| Choc conjoncturel inédit | La promotion 2020 est systématiquement surestimée |

Le score global est optimiste : {RELEVES_ETAPE_5['recouvrement_formations']:.0f} % des
formations du jeu de test apparaissent aussi à l'entraînement, par une autre promotion.

## Limites d'usage

1. **Le modèle décrit, il n'explique pas.** Une variable importante n'est pas une cause. Le
   type de diplôme prédit bien parce qu'il résume un public, une discipline et un marché.
2. **Variables contemporaines de la cible.** Les effectifs de sortants et de poursuivants
   proviennent de la même enquête à six mois : le modèle prédit au moment de l'enquête, pas
   avant la sortie de promotion.
3. **Seuil de publication.** Aucun taux n'est publié sous 20 sortants : les petites
   formations sont absentes des données, et donc hors du domaine de validité.
4. **Un taux d'emploi salarié n'est pas un taux d'insertion.** L'emploi non salarié est
   mesuré séparément et exclu.

## Ce pour quoi ce modèle ne doit pas être utilisé

- **Décider du financement ou de la fermeture d'une formation.** L'erreur moyenne dépasse
  9 points et double sur les petites formations : elle est du même ordre que les écarts qu'on
  voudrait arbitrer.
- **Comparer deux formations proches.** Un écart de 5 points prédits est dans le bruit.
- **Évaluer une personne.** L'unité de prédiction est une formation entière.
- **Classer des établissements.** Un établissement spécialisé hérite de la performance de
  son secteur, l'étape 2 le montre.

## Biais connus

- **Biais de sélection** : les formations sous 20 sortants sont exclues par la source. Les
  petites formations, souvent rurales ou spécialisées, ne sont pas représentées.
- **Biais de représentation** : 58 % des lignes du fichier brut n'ont pas de cible et
  disparaissent du périmètre.
- **Biais d'interprétation** : le taux d'emploi salarié pénalise mécaniquement les formations
  qui mènent à une poursuite d'études. Un taux bas n'y signale aucun échec.
- **Angle mort** : genre, nationalité et régime d'inscription ne sont pas dans le modèle. Il
  ne peut rien dire des disparités, et ne doit pas être invoqué pour les nier.

## Utilisation

```python
from modeles.inference import charger, predire

modele = charger()
predire(modele, {{
    'Région': 'Bretagne',
    'Académie': 'Rennes',
    'Type de diplôme': 'Licence professionnelle',
    'Domaine disciplinaire': 'Sciences, technologies, santé',
    'Discipline': 'Sciences fondamentales et applications',
    'Secteur disciplinaire': 'Informatique',
    'Promotion': 2024,
    "Code UAI de l'établissement": '0350936C',
    'Code du diplôme SISE': '99999',
    '6-Nombre de sortants - 6 mois après le diplôme': 48,
    '6-Nombre de poursuivants - 6 mois après le diplôme': 6,
}})
```

Une modalité inconnue ne fait pas échouer la prédiction : la variable concernée retombe sur
la médiane d'apprentissage, et la clé `replis` de la réponse dit lesquelles.

## Maintenance

Le modèle est lié au millésime 2026_S1. À chaque nouveau millésime InserSup, réexécuter les
notebooks des étapes 1 à 5 dans l'ordre, puis ce notebook : les contrôles de conformité
échouent si une fuite ou un identifiant s'est glissé dans les variables.
"""

CARTE_PATH.write_text(carte, encoding='utf-8')
print(f'Fiche écrite : {CARTE_PATH.relative_to(PROJECT_PATH)} '
      f'({len(carte.splitlines())} lignes)')
print()
print('\n'.join(carte.splitlines()[:18]))

Fiche écrite : MODEL_CARD.md (114 lignes)

# Fiche du modèle : insertion professionnelle des formations

*Fiche générée par [`etape_7_synthese_modele.ipynb`](notebooks/etape_7_synthese_modele.ipynb)
à partir du modèle livré. Ne pas modifier à la main.*

## Ce que fait ce modèle

Il prédit le **taux d'emploi salarié en France six mois après le diplôme** d'une formation
de l'enseignement supérieur, à partir de 19 caractéristiques connues
avant l'enquête d'insertion.

- **Tâche** : régression, cible continue de 0 à 100 %.
- **Algorithme** : RandomForestRegressor, 200 arbres,
  `min_samples_leaf=2`, `random_state=42`.
- **Unité de prédiction** : une formation, c'est-à-dire un diplôme d'un établissement pour
  une promotion donnée. **Jamais une personne.**

## Données d'entraînement


---

## Checklist de la phase 7

**7.1 Rigueur méthodologique**

- [x] `train_test_split` réalisé avant tout prétraitement
- [x] `stratify` sans objet : régression
- [x] Tout le prétraitement dans un `ColumnTransformer` / `Pipeline`
- [x] Aucune colonne fuitante dans `X`, identifiants exclus, vérifié par `assert` ci-dessus
- [x] Le jeu de test n'a servi qu'une fois, à l'étape 5
- [x] `random_state` fixé partout, et contrôlé sur le modèle livré

**7.2 Baseline et comparaison**

- [x] `DummyRegressor` calculé dans les mêmes conditions que les modèles
- [x] 6 modèles comparés, dont la régression linéaire et Ridge
- [x] `GridSearchCV` avec `cv=5`, tableau `cv_results_` commenté à l'étape 5
- [x] Choix final justifié par la règle à un écart-type, et son coût chiffré

**7.3 Évaluation**

- [x] Métrique principale justifiée par le métier : la MAE, en points de taux d'emploi
- [x] MAE, RMSE dans l'unité de la cible, R² reporté, résidus analysés
- [x] Écart entraînement / validation traduit en nombre de formations
- [x] Score de test reporté honnêtement, gain sur la référence naïve chiffré

**7.4 Interprétation et limites**

- [x] Importance des variables par deux méthodes, impureté et permutation
- [x] Cohérence avec l'EDA discutée, y compris les deux hypothèses infirmées
- [x] Cas d'échec identifiés et mesurés : petites formations, formations inconnues, 2020
- [x] Biais de sélection, de représentation et d'interprétation discutés dans la fiche

## Utilisation de l'IA sur cette étape

| Prompt utilisé | Ce que l'IA a produit | Vérification effectuée |
|---|---|---|
| « Que contient une bonne fiche modèle ? » | Structure en sections, dont « usage prévu » et « hors périmètre » | Retenue, mais la section la plus utile, « ce pour quoi ce modèle ne doit pas être utilisé », a été écrite à partir des erreurs mesurées à l'étape 5, pas d'un modèle générique |
| « Écris des tests pour vérifier l'absence de fuite dans un modèle sauvegardé » | Boucle sur les noms de colonnes avec motifs interdits | Conservée, mais complétée : un test de déterminisme et un contrôle du `random_state`, que la proposition initiale ne couvrait pas |
| « Mon modèle peut-il servir à décider de fermer une formation ? » | Réponse nuancée sur les usages à haut risque | Traduite en critère chiffré : l'erreur moyenne est du même ordre que les écarts à arbitrer, donc non |

Le troisième échange est celui qui a le plus changé le livrable : il a transformé une
intuition, « attention à l'usage », en une limite argumentée par les chiffres du projet.